In [2]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv, dotenv_values

load_dotenv(override=True)
env_dict = dotenv_values(".env")
# print("--- Contents of .env file ---")
# for key, value in env_dict.items():
#     # Masking the value so you don't accidentally print your full API key to your screen!
#     masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
#     print(f"{key}: {masked_value}")

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_11868/4189044228.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser

#data model
class RouteQuery(BaseModel):
    """ Route a user query to the most relevant datasource """
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
    ...,
    description="Given a user question choose which datasource would be most relevant for answering their question",
    )   

llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", temperature=0
)
structured_llm = llm.with_structured_output(RouteQuery)

#prompt 
# Prompt 
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

#define router
router = prompt | structured_llm

In [14]:
from langchain_core.output_parsers import PydanticOutputParser

# 1. Create the parser
parser = PydanticOutputParser(pydantic_object=RouteQuery)

# 2. Inject the format instructions directly into the system prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source.

{format_instructions}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

# 3. Build the chain (Prompt -> Standard LLM -> Parser)
# Notice we are using standard `llm` here, not `with_structured_output`
router = prompt | llm | parser

# 4. Invoke as normal
result = router.invoke({"question": question})

In [15]:

question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([("human", "speak in {language}")])
prompt.invoke({"language": "french"})
"""

result = router.invoke({"question": question})

In [16]:
result

RouteQuery(datasource='python_docs')

In [17]:
result.datasource


'python_docs'

In [18]:
def choose_route(result):
    if 'python_docs' in result.datasource.lower():
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        return "chain for js_docs"
    else:
        return "golang_docs"

from langchain_core.runnables import RunnableLambda
full_chain = router | RunnableLambda(choose_route)


In [19]:
result = full_chain.invoke({"question": question})

In [20]:
print(result)

chain for python_docs


In [31]:
#semnatic routing
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

#embded prompts 
embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1", 
    api_key="lm-studio", 
    model="text-embedding-nomic-embed-text-v1.5", # Explicitly set your local model name here
    check_embedding_ctx_length=False 
)
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

#route question to prompt 
def prompt_router(input):
    #embed question
    query_embedding = embeddings.embed_query(input["query"])
    #compute similarity
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    #chosen prompt
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", model = "google/gemma-4-e4b")
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

Using PHYSICS
Ah, a fundamental question in modern astrophysics! I'd be delighted to explain it for you.

Simply put, **a black hole is a region of spacetime where gravity is so incredibly strong that nothing—not even light—can escape from it.**

Here is a more detailed breakdown:

### 🌌 The Mechanics

1.  **Formation:** Black holes typically form when massive stars (much larger than our Sun) run out of fuel and undergo a catastrophic gravitational collapse. The stellar core implodes under its own immense gravity.
2.  **The Singularity:** All the matter from the collapsed star is crushed down into an infinitely small, dense point called a *singularity*. This singularity contains almost all the mass.
3.  **Gravity's Edge (The Event Horizon):** Because the mass is so concentrated, it warps spacetime severely. The boundary around the black hole—the **Event Horizon**—is the critical threshold. If you cross this point, the escape velocity needed to leave is greater than the speed of light, 

In [23]:
#query structuring 
from pprint import pprint
# query structuring 
from langchain_community.document_loaders import YoutubeLoader

# 1. Set add_video_info to False. This safely bypasses the broken YouTube scraper
# and successfully downloads the text transcript using the stable youtube-transcript-api.
docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=pbAd8O1Lvm4",
    add_video_info=False
).load()

pprint.pprint(docs[0].metadata)

IpBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=pbAd8O1Lvm4! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

Ways to work around this are explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).


If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide th

In [22]:
import datetime
from typing import Literal, Optional, Tuple
from pydantic import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video "
            "title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None,
        description="Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

llm = ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", model = "google/gemma-4-e4b")
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm


In [26]:
query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()


content_search: rag from scratch tutorial
title_search: *rag* *from scratch*


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None


In [27]:
query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
).pretty_print()

content_search: langchain AND "chat" AND YEAR(publication) = 2023
title_search: chat langchain 2023


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None


In [28]:

query_analyzer.invoke(
    {"question": "videos that are focused on the topic of chat langchain that are published before 2024"}
).pretty_print()

content_search: chat langchain < 2024
title_search: chat langchain
latest_publish_date: 2023-12-31


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None


In [29]:
query_analyzer.invoke(
    {
        "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
    }
).pretty_print()


content_search: tutorial_videos
title_search: multi-modal models in an agent
min_length_sec: 0
max_length_sec: 300


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:46: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for field in self.__fields__:
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_73814/3806102375.py:48: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  self.__fields__[field], "default", None
